# Día 8 · Unidad 8 — Visualización (II): seaborn — distribución y categóricos

**Objetivos de aprendizaje**
- Usar `sns.histplot()`, `sns.jointplot()` y `sns.pairplot()` para explorar la distribución de variables numéricas.
- Visualizar el Teorema del Límite Central con datos reales del curso.
- Usar `sns.countplot()`, `sns.barplot()`, `sns.boxplot()`, `sns.violinplot()` y `sns.stripplot()`/`sns.swarmplot()` para variables categóricas.
- **Pedir a la IA que genere un gráfico y revisar manualmente el código** — el segundo bloque de IA del día, esta vez sobre visualización.

**Duración estimada:** 100-120 minutos.

**Requisitos previos:** `03_teoria_matplotlib_basico.ipynb` de este mismo día.

## 1. Qué es seaborn

[seaborn](https://seaborn.pydata.org/) es una librería de visualización estadística construida **sobre** Matplotlib: da una interfaz de alto nivel (mucho menos código) para gráficos que, en Matplotlib puro, requerirían bastante trabajo manual. Por dentro, casi todo lo que dibuja seaborn sigue siendo `Figure`/`Axes` de Matplotlib — lo que aprendisteis en el notebook anterior sigue aplicando (`ax.set_title()`, `fig.tight_layout()`...).

Trabajaremos con `siniestros.csv`, y en algún ejemplo lo combinaremos con `cartera_polizas.csv` mediante el `.merge()` que visteis en el notebook `02` de hoy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

siniestros = pd.read_csv("../data/raw/siniestros.csv", parse_dates=["fecha_siniestro"])
cartera = pd.read_csv("../data/raw/cartera_polizas.csv")

siniestros.head()

## 2. Distribution plots

### `sns.histplot()`

Histograma (y, opcionalmente, estimación de densidad *KDE*) de una variable numérica.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=siniestros, x="importe_pagado", kde=True, bins=30, ax=ax)
ax.set_title("Distribución del importe pagado por siniestro")
ax.set_xlabel("Importe pagado (€)")
plt.show()

La distribución tiene una cola larga hacia la derecha (hay pocos siniestros muy caros, pero no siniestros "muy baratos" en la misma proporción) — el mismo patrón de asimetría que ya visteis con `importe` en los ejercicios del Día 7.

### `sns.jointplot()`

Empareja dos variables numéricas: el gráfico central (scatter, por defecto) más un histograma/densidad de cada variable en los márgenes. Traemos `edad_tomador` desde `cartera` con el `.merge()` que aprendisteis hoy.

In [ ]:
siniestros_con_edad = siniestros.merge(
    cartera[["poliza_id", "edad_tomador", "producto"]],
    on="poliza_id",
    how="left",
    suffixes=("", "_cartera"),
)

plot = sns.jointplot(
    data=siniestros_con_edad,
    x="edad_tomador",
    y="importe_pagado",
    kind="scatter",
    height=6,
)
plot.fig.suptitle("Importe pagado vs. edad del tomador", y=1.02)
plt.show()

### `sns.pairplot()`

Muestra la relación de **todas las variables numéricas por pares** a la vez — el punto de partida habitual al explorar un dataset nuevo, antes de decidir en qué relación concreta profundizar.

In [ ]:
numericas_siniestro = siniestros_con_edad[["edad_tomador", "importe_pagado", "producto"]].dropna()

plot = sns.pairplot(data=numericas_siniestro, hue="producto", diag_kind="hist", height=2.8)
plot.fig.suptitle("Relación entre variables numéricas, por producto", y=1.02)
plt.show()

> 📊 **En Excel esto sería...** como generar automáticamente un panel completo de gráficos de dispersión para cada par de columnas numéricas de una tabla — algo que en Excel tocaría montar gráfico a gráfico, a mano, uno por combinación.

## 3. Teorema del Límite Central

El [Teorema del Límite Central](https://es.wikipedia.org/wiki/Teorema_del_l%C3%ADmite_central) dice que la distribución de una **media** de variables aleatorias independientes se aproxima a una normal a medida que crece el tamaño de la muestra que se promedia — aunque la variable original (aquí, `importe_pagado`) no sea nada normal por sí sola, como acabáis de ver arriba. Es la base de los supuestos de muchos métodos estadísticos y de machine learning.

Comprobémoslo con los propios datos: el importe medio por `tipo_siniestro`, ponderado por cuántas observaciones tiene cada tipo, debería parecerse más a una normal que la distribución bruta de `importe_pagado`.

In [ ]:
resumen_tipo = siniestros.groupby("tipo_siniestro").agg(
    importe_medio=("importe_pagado", "mean"),
    n=("importe_pagado", "count"),
)
resumen_tipo

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(data=siniestros, x="importe_pagado", ax=ax1)
ax1.set_title("Importe pagado (dato bruto): asimétrico")

sns.kdeplot(data=resumen_tipo, x="importe_medio", weights="n", ax=ax2)
ax2.set_title("Importe medio por tipo de siniestro,\nponderado por nº de casos: más simétrico")

fig.tight_layout()
plt.show()

No es una prueba formal (haría falta un test de normalidad de verdad, que veréis en la Unidad 10), pero la diferencia visual entre ambos paneles es justamente la intuición que el Teorema del Límite Central formaliza: **medias de variables se comportan mucho "mejor" que las variables originales.**

**Para ti:** repetid el `histplot` de `importe_pagado`, pero filtrando solo `producto == "AUTO"`. ¿La forma de la distribución cambia mucho respecto a la de todos los productos juntos?

In [ ]:
# Tu código aquí


## 4. Categorical plots

### `sns.countplot()`

El histograma de las variables categóricas: cuenta observaciones por categoría.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=siniestros, y="tipo_siniestro", order=siniestros["tipo_siniestro"].value_counts().index, ax=ax)
ax.set_title("Número de siniestros por tipo")
ax.set_xlabel("Número de siniestros")
ax.set_ylabel("Tipo de siniestro")
plt.show()

Usamos `y=` en vez de `x=` para que las categorías (con nombres largos como `asistencia_en_carretera`) se lean en horizontal, y `order=` para ordenarlas de mayor a menor en vez de dejarlas en el orden en que aparecen en los datos — sin esto, un gráfico de barras categórico es mucho más difícil de leer de un vistazo.

### `sns.barplot()`

Representa una función de agregación (la media, por defecto) de una variable numérica por categoría, con una medida de incertidumbre.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=siniestros, x="producto", y="importe_pagado", estimator="mean", ax=ax)
ax.set_title("Importe medio pagado por producto")
ax.set_ylabel("Importe medio (€)")
plt.show()

### `sns.boxplot()`

Muestra la distribución completa (mediana, cuartiles, outliers) de una variable numérica por categoría — más información que un `barplot`, que solo muestra la media.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=siniestros, x="producto", y="importe_pagado", ax=ax)
ax.set_title("Distribución del importe pagado por producto")
plt.show()

### `sns.violinplot()`

Como el `boxplot`, pero mostrando la forma completa de la densidad estimada, no solo cuartiles.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.violinplot(
    data=siniestros,
    x="producto",
    y="importe_pagado",
    hue="estado",
    split=True,
    density_norm="count",  # escala el ancho por nº de observaciones de cada grupo
    ax=ax,
)
ax.set_title("Distribución del importe pagado por producto y estado del siniestro")
plt.show()

> ⚠️ **Error típico**: usar el parámetro `scale="count"` en `violinplot()`. Es el nombre que tenía en versiones antiguas de seaborn (y así aparece en mucha documentación y tutoriales por internet), pero en **seaborn 0.13** (la versión instalada en este curso, ver `requirements.txt`) se renombró a `density_norm="count"`. Si escribís `scale="count"` con esta versión, seaborn lanza un `TypeError` porque el parámetro ya no existe — es exactamente el tipo de cosa que conviene comprobar contra la versión instalada, no contra la documentación o un ejemplo que encontréis sueltos, cuando algo que "debería funcionar" no funciona.

In [ ]:
try:
    sns.violinplot(data=siniestros, x="producto", y="importe_pagado", scale="count")
except TypeError as e:
    print("Error esperado (parámetro obsoleto en seaborn 0.13):", e)
plt.close("all")

### `sns.stripplot()` y `sns.swarmplot()`

Un `scatterplot` donde una de las variables es categórica: cada punto es una observación individual, sin agregar nada. `swarmplot` es la variante que evita que los puntos se solapen (más lenta con datasets grandes).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=siniestros, x="producto", y="importe_pagado", boxprops={"alpha": 0.3}, ax=ax)
sns.stripplot(data=siniestros, x="producto", y="importe_pagado", alpha=0.4, ax=ax)
ax.set_title("Importe pagado por producto: caja + observaciones individuales")
plt.show()

Combinar `boxplot` (con transparencia) y `stripplot` en el mismo `Axes` es un patrón muy útil: la caja da el resumen estadístico, los puntos dejan ver la cantidad real de datos y si hay agrupaciones que el resumen por sí solo esconde.

**Para ti:** haced un `sns.boxplot()` de `importe_pagado` por `estado` (`abierto`/`cerrado`). ¿Los siniestros abiertos tienden a tener importes distintos de los cerrados, o se parecen?

In [ ]:
# Tu código aquí


## 5. Con ayuda de la IA: gráficos asistidos — revisión manual del código

Pedirle a un asistente de IA que genere un gráfico es rápido, pero un gráfico puede ser **técnicamente correcto y aun así engañoso**: un eje Y que no empieza en cero exagera visualmente las diferencias, una paleta de colores mal elegida puede sugerir un orden que no existe, o el tipo de gráfico puede no ser el más honesto para el dato que se quiere comunicar. Aquí el bloque de IA no consiste en validar un número (como en el notebook anterior), sino en **revisar el código del gráfico con criterio antes de publicarlo**.

### Prompt sugerido

In [ ]:
prompt_grafico_asistido = """
Contexto: tengo un DataFrame de pandas 'siniestros' con columnas
'tipo_siniestro' (categorica) e 'importe_pagado' (numerica). Quiero un
grafico de barras que compare el importe medio pagado por tipo de
siniestro.

Objetivo: dame el codigo con seaborn/matplotlib (enfoque stateless, con
fig, ax = plt.subplots()) para un barplot del importe medio por tipo de
siniestro, ordenado de mayor a menor, con titulo y etiquetas de eje.

Restricciones: usa una paleta de colores que tenga sentido para datos
categoricos SIN orden inherente (no una paleta secuencial tipo "un color
por intensidad", porque tipo_siniestro no tiene un orden natural).

Formato de salida: solo el bloque de codigo, sin explicacion adicional.
"""
print(prompt_grafico_asistido)

**Qué se espera obtener**: un bloque de código con `sns.barplot(...)`, `order=` para ordenar de mayor a menor, y una paleta cualitativa (categorías sin orden) en vez de secuencial (que sugeriría una escala de intensidad que no existe aquí).

**Qué debes verificar antes de darlo por bueno** — no ejecutéis el gráfico y deis por bueno que "se ve bien"; repasad el código con esta checklist, **antes** de mirar el resultado:

- [ ] **¿El eje Y empieza en cero?** Un `barplot`/`countplot` cuya área representa una cantidad (barras) engaña si el eje no arranca en cero: una diferencia real del 5% puede *parecer* del 300% si el eje empieza en, por ejemplo, 400 en vez de 0. Matplotlib/seaborn lo hacen bien por defecto en un `barplot`, pero conviene comprobarlo explícitamente, sobre todo si alguien ha tocado `ax.set_ylim(...)` a mano.
- [ ] **¿La paleta de colores es adecuada al tipo de dato?** Una paleta *secuencial* (tonos de un mismo color, de claro a oscuro) solo tiene sentido si las categorías tienen un orden real (por ejemplo, niveles de riesgo bajo/medio/alto). Para categorías sin orden (como `tipo_siniestro`), lo correcto es una paleta *cualitativa* (colores distintos, no un degradado) — usar una secuencial ahí sugiere visualmente un orden o intensidad que no existe en los datos.
- [ ] **¿El tipo de gráfico es el correcto para lo que se quiere comunicar?** Un `barplot` comunica bien una comparación de medias entre categorías; si lo que interesa es la **distribución completa** (no solo la media), un `barplot` esconde información que un `boxplot`/`violinplot` sí mostraría — dos tipos de siniestro pueden tener la misma media y una dispersión completamente distinta.

In [ ]:
# Un ejemplo de código GENERADO POR IA (simulado aquí para el notebook) con un fallo típico:
# la paleta secuencial ("Blues") sugiere un orden de intensidad que tipo_siniestro no tiene.
orden_por_media = siniestros.groupby("tipo_siniestro")["importe_pagado"].mean().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(
    data=siniestros, y="tipo_siniestro", x="importe_pagado",
    order=orden_por_media, palette="Blues", ax=ax,  # <- paleta secuencial: no encaja con categorías sin orden
)
ax.set_title("Versión con fallo: paleta secuencial en categorías sin orden real")
plt.show()

Con la paleta `"Blues"`, el `tipo_siniestro` con la barra más oscura parece "más importante" o "el final de una escala" — pero el orden de la lista (`orden_por_media`) es simplemente de mayor a menor importe medio, no una escala continua inherente al tipo de siniestro. La corrección es usar una paleta cualitativa:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(
    data=siniestros, y="tipo_siniestro", x="importe_pagado",
    order=orden_por_media, palette="tab10", ax=ax,  # <- paleta cualitativa: colores distintos, sin degradado
)
ax.set_title("Versión corregida: paleta cualitativa (categorías sin orden)")
ax.set_xlabel("Importe medio pagado (€)")
ax.set_ylabel("Tipo de siniestro")

# Confirmamos explícitamente que el eje X (aquí es el de la magnitud, por ser barras horizontales) empieza en 0
print("Límite inferior del eje de magnitud:", ax.get_xlim()[0])
assert ax.get_xlim()[0] <= 0, "El eje de magnitud de un gráfico de barras debe empezar en 0 (o menos)"
plt.show()

El `assert` sobre `ax.get_xlim()` es la forma de hacer explícita y verificable la primera pregunta de la checklist ("¿el eje empieza en cero?"), en vez de fiaros de un vistazo rápido — el mismo principio de validación que en el resto del curso, aplicado ahora a un gráfico en vez de a un número.

## Resumen

- seaborn es una capa de alto nivel sobre Matplotlib — por dentro sigue devolviendo (o aceptando) objetos `Axes`/`Figure`, así que todo lo del notebook anterior sigue aplicando.
- Distribution plots (`histplot`, `jointplot`, `pairplot`) exploran variables numéricas y su relación entre sí — buen punto de partida al conocer un dataset nuevo.
- El Teorema del Límite Central se puede visualizar comparando la distribución bruta de una variable con la distribución de sus medias por grupo, ponderadas por tamaño de grupo.
- Categorical plots: `countplot` (conteo), `barplot` (media + incertidumbre), `boxplot`/`violinplot` (distribución completa), `stripplot`/`swarmplot` (observaciones individuales) — cada uno comunica algo distinto, y elegir el equivocado puede ocultar información relevante.
- `violinplot(..., scale="count")` ya no existe en seaborn 0.13: el parámetro se renombró a `density_norm="count"` — comprobad siempre contra la versión instalada (`requirements.txt`), no contra ejemplos sueltos de internet.
- Un gráfico generado por IA (o por cualquiera) se revisa con criterio antes de publicarlo: ¿el eje empieza en cero cuando debería?, ¿la paleta encaja con el tipo de dato (cualitativa para categorías sin orden, secuencial solo si hay una escala real)?, ¿el tipo de gráfico elegido comunica lo que de verdad hace falta (media, distribución completa, observaciones individuales)?

**Siguiente:** `05_teoria_matrix_plots_heatmap.ipynb`, donde cerramos la Unidad 8 con matrices de correlación, `heatmap` y `clustermap`.